# FINE TUNING


### Imports


In [ ]:
import os
import Path
import numpy as np
import matplotlib.pyplot as plt
import random
import json

from batch import Batch
from collections import Counter
from dotenv import load_dotenv
from huggingface_hub import login
from contracts import ShippingContract
from loaders import ShippingContractLoader
from evaluator import evaluate
from openai import OpenAI
from tenacity import retry, retry_if_result, wait_fixed

load_dotenv(override=True)

### Hugging face login


In [ ]:
hf_token: str = os.environ["HF_TOKEN"]
login(token=hf_token, add_to_git_credential=True)

### Load Data


In [ ]:
dataset_names = ["train"]
contracts = ShippingContractLoader().load()

In [ ]:
# remove duplicates if any
seen = set()
contracts: list[ShippingContract] = [
    contract
    for contract in contracts
    if not (contract.contract_number in seen or seen.add(contract.contract_number))
]

print(f"There are {len(contracts)} contracts now in the dataset")


def plot_shipping_costs(shipping_costs):
    # Plot the distribution of shipping costs
    plt.figure(figsize=(15, 6))
    plt.title(
        f"Shipping Costs: Avg {sum(shipping_costs) / len(shipping_costs):,.2f} and highest {max(shipping_costs):,}\n"
    )
    plt.xlabel("Shipping Cost ($)")
    plt.ylabel("Count")
    plt.hist(shipping_costs, rwidth=0.7, color="orange", bins=range(0, 100000, 1000))
    plt.show()


def plot_categories(categories, counts):
    """
    Show the distribution of categories in the dataset
    """
    plt.figure(figsize=(15, 6))

    plt.bar(categories, counts, color="goldenrod")
    plt.title("How many in each category")
    plt.xlabel("Categories")
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")

    for i, v in enumerate(counts):
        plt.text(i, v, f"{v:,}", ha="center", va="bottom")

    plt.show()

In [ ]:
shipping_costs = [contract.total_shipping_cost for contract in contracts]

plot_shipping_costs(shipping_costs)

In [ ]:
category_counts = Counter([contract.goods_description.value for contract in contracts])
categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plot_categories(categories, counts)

In [ ]:
min_price = min(contracts, key=lambda x: x.total_shipping_cost).total_shipping_cost
max_price = max(contracts, key=lambda x: x.total_shipping_cost).total_shipping_cost

min_index = next(
    i for i, c in enumerate(contracts) if c.total_shipping_cost == min_price
)
max_index = next(
    i for i, c in enumerate(contracts) if c.total_shipping_cost == max_price
)

print(min_price, min_index)
print(max_price, max_index)


In [ ]:
print(contracts[min_index].full)

In [ ]:
print(contracts[max_index].full)

In [ ]:
np.random.seed(42)

SIZE = 160

prices = np.array([contract.total_shipping_cost for contract in contracts], dtype=float)
categories = np.array([contract.goods_description.value for contract in contracts])
p = (prices - prices.min()) / (prices.max() - prices.min())

w = p
w[categories == "Electronics"] *= 0.05
w[categories == "Machinery"] *= 0.05
w[categories == "Toys"] *= 0.05
w[categories == "Pharmaceuticals"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(contracts), size=SIZE, replace=False, p=w)
sample = [contracts[i] for i in idx]


In [ ]:
prices = [contract.total_shipping_cost for contract in sample]
plot_shipping_costs(prices)

In [ ]:
random.seed(42)
random.shuffle(sample)

prices = [contract.total_shipping_cost for contract in sample]
plot_shipping_costs(prices)


In [ ]:
category_counts = Counter([contract.goods_description.value for contract in sample])
categories = category_counts.keys()
counts = [category_counts[category] for category in categories]

plot_categories(categories, counts)


In [ ]:
def plot_categories_pie(categories, counts):
    plt.figure(figsize=(12, 10))

    plt.pie(counts, labels=categories, autopct="%1.0f%%", startangle=90)

    # Add a circle at the center to create a donut chart (optional)
    centre_circle = plt.Circle((0, 0), 0.70, fc="white")
    fig = plt.gcf()
    fig.gca().add_artist(centre_circle)
    plt.title("Categories")

    # Equal aspect ratio ensures that pie is drawn as a circle
    plt.axis("equal")

    plt.show()


plot_categories_pie(categories, counts)

In [ ]:
def plot_price_vs_weight_corelation(sample):
    # How does the shipping cost vary with the weight?

    weights = [contract.total_weight for contract in sample]
    prices = [contract.total_shipping_cost for contract in sample]

    # Create the scatter plot
    plt.figure(figsize=(15, 8))
    plt.scatter(weights, prices, s=0.2, color="red")

    # Add labels and title
    plt.xlabel("Weight in kgs")
    plt.ylabel("Price in USD")
    plt.title("Is there a simple correlation with weight?")

    # Display the plot
    plt.show()


plot_price_vs_weight_corelation(sample)

### Push To hugging face


In [ ]:
username = "gathondu"
full = f"{username}/supply-chain-contracts-full"

train: list[ShippingContract] = sample[:128]
val: list[ShippingContract] = sample[128:144]
test: list[ShippingContract] = sample[144:]

ShippingContract.push_to_hub(full, train, val, test)


## Data Pre Processing

### Pull dataset from Hugging face, preprocess it and then push it back to hugging face


In [ ]:
dataset = f"{username}/supply-chain-contracts-full"

train, val, test = ShippingContract.from_hub(dataset)

contracts = train + val + test

Batch.create(contracts)
Batch.run()
Batch.fetch()
print(contracts[110].summary)
train: list[ShippingContract] = contracts[:128]
val: list[ShippingContract] = contracts[128:144]
test: list[ShippingContract] = contracts[144:]

preprocessed = f"{username}/supply-chain-contracts-preprocessed"

ShippingContract.push_to_hub(preprocessed, train, val, test)

## Fine tune frontier model (gpt-4.1-nano-2025-04-14)


In [ ]:
dataset = f"{username}/supply-chain-contracts-preprocessed"

train, val, test = ShippingContract.from_hub(dataset)
print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items"
)

openai = OpenAI()


def messages_for(contract):
    message = f"Estimate the shipping cost for this contract. Respond with the shipping cost, no explanation\n\n{contract.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${contract.total_shipping_cost} USD"},
    ]


def make_jsonl(contracts):
    result = ""
    for contract in contracts:
        messages = messages_for(contract)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + "}\n"
    return result.strip()


def write_jsonl(contracts, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(contracts)
        f.write(jsonl)


Path("jsonl").mkdir(parents=True, exist_ok=True)
write_jsonl(train, "jsonl/fine_tune_train.jsonl")
write_jsonl(val, "jsonl/fine_tune_validation.jsonl")

with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    hyperparameters={
        "n_epochs": 1,
        "batch_size": 1,
    },
    suffix="supply-chain",
)

first_job = next(iter(openai.fine_tuning.jobs.list(limit=1)))
job_id = first_job.id

job = openai.fine_tuning.jobs.retrieve(job_id)


@retry(retry=retry_if_result(lambda result: not result), wait=wait_fixed(10))
def wait_for_job(job_id):
    job = openai.fine_tuning.jobs.retrieve(job_id)
    print(job.status)
    return job.status == "succeeded"


wait_for_job(job_id)
fine_tuned_model_name = job.fine_tuned_model
print(fine_tuned_model_name)


def test_messages_for(contract):
    message = f"Estimate the shipping cost for this contract. Respond with the shipping cost, no explanation\n\n{contract.summary}"
    return [
        {"role": "user", "content": message},
    ]


def gpt_4__1_nano_fine_tuned(contract):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name, messages=test_messages_for(contract), max_tokens=7
    )
    return response.choices[0].message.content


evaluate(gpt_4__1_nano_fine_tuned, test)